In [1]:
%run init_notebook.py
import torch

SRATE = 12000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch_directml.device()
print(f"Using device: {device}")


Using device: cuda


# Data Augmentation for Neural Sound Synthesis:

Soruce: https://github.com/iver56/torch-audiomentations

Data augmentation is used to increase the diversity of training data without actually collecting new data. 

### Common techniques include:
1. **Pitch Shifting**: Changing the pitch of the audio without affecting its duration.
2. **Time Stretching**: Changing the duration of the audio without affecting its pitch.
3. **Adding Reverb**: Simulating different acoustic environments by adding reverberation.
4. **Filtering**: Applying various filters (e.g., low-pass, high-pass) to alter the frequency content of the audio.
5. **Polarity Inversion**: Inverting the audio signal to create a new version of the sound.
6. **Polyphonic Mixing**: Combining multiple audio samples to create a new, more complex sound.
7. **Compression and Distortion**: Applying dynamic range compression or distortion effects to alter the sound characteristics.
8. **Combination of Techniques**: Using multiple augmentation techniques together to create even more diverse training data.


In [2]:
from torch_audiomentations import Compose, PitchShift, LowPassFilter, HighPassFilter, BandPassFilter, PolarityInversion

mode = "per_example"  # aplicar una transformacion diferente a cada ejemplo del batch
p_mode = "per_example" # la probabilidad de aplicar cada transformacion se decide de forma independiente para cada ejemplo del batch
p = 0.5 # probabilidad de aplicar cada transformacion # TODO ver que valor poner aqui

# PitchShift
ps_p = 0.5
min_transpose_semitones = -4
max_transpose_semitones = 4
# LowPassFilter
lp_p = 0.5
min_cutoff_freq = 300.0
max_cutoff_freq = 6000.0
# HighPassFilter
hp_p = 0.5
min_cutoff_freq = 300.0
max_cutoff_freq = 6000.0
# BandPassFilter
bp_p = 0.5
min_center_freq = 300.0
max_center_freq = 3000.0
min_q_factor = 0.6
max_q_factor = 1.9
# PolarityInversion
pi_p = 0.5

batch_size = 2**6

apply_augmentation = Compose(
    transforms=[
        PitchShift(mode=mode, p=ps_p, p_mode=p_mode, sample_rate=SRATE, min_transpose_semitones=min_transpose_semitones, max_transpose_semitones=max_transpose_semitones),
        LowPassFilter(mode=mode, p=lp_p, p_mode=p_mode, sample_rate=SRATE, min_cutoff_freq=min_cutoff_freq, max_cutoff_freq=max_cutoff_freq),
        HighPassFilter(mode=mode, p=hp_p, p_mode=p_mode, sample_rate=SRATE, min_cutoff_freq=min_cutoff_freq, max_cutoff_freq=max_cutoff_freq),
        BandPassFilter(mode=mode, p=bp_p, p_mode=p_mode, sample_rate=SRATE, min_center_frequency=min_center_freq, max_center_frequency=max_center_freq, min_bandwidth_fraction=min_q_factor, max_bandwidth_fraction=max_q_factor),
        PolarityInversion(mode=mode, p=pi_p, p_mode=p_mode)
    ],
    shuffle=True
)

c:\Users\Articuno\Desktop\TFG-MUSICAL\env-musical\Lib\site-packages\torch_audiomentations\core\transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = PitchShift(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
c:\Users\Articuno\Desktop\TFG-MUSICAL\env-musical\Lib\site-packages\torch_audiomentations\core\transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = LowPassFilter(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
c:\Users\Articuno\Desktop\TFG-MUSICAL\env-musical\Lib\site-packages\torch_audiomentatio

In [3]:
from src.paths import PATHS
import os

# Make an example tensor with white noise.
# This tensor represents 8 audio snippets with 1 channel (mono) and 2 s of 16 kHz audio.
audio_samples = torch.rand(size=(8, 1, 32000), dtype=torch.float32, device=device) - 0.5 # TODO load real audio samples

# Apply augmentation. This varies the gain and polarity of (some of)
# the audio snippets in the batch independently.
perturbed_audio_samples = apply_augmentation(audio_samples, sample_rate=SRATE)
print("Original audio samples shape:", audio_samples.shape)
print("Perturbed audio samples shape:", perturbed_audio_samples.shape)
# Save the new augmented audio samples to disk 

path = PATHS['augmented_data_1']
for i in range(perturbed_audio_samples.shape[0]):
    _p = os.path.join(path, f"augmented_sample_{i}.wav")
    # torchaudio.save(w, perturbed_audio_samples[i], SRATE)

Original audio samples shape: torch.Size([8, 1, 32000])
Perturbed audio samples shape: torch.Size([8, 1, 32000])


In [4]:
import os
import torchaudio
from torch.utils.data import DataLoader
from src.dataset import NSynth
from tqdm import tqdm

train_loader = DataLoader(NSynth('training'), batch_size=batch_size, shuffle=True, pin_memory=True)
pbar = tqdm(train_loader, unit="batch")
path = PATHS['augmented_data_1']

os.makedirs(path, exist_ok=True)

for j, (ws, p1, p2, p3) in enumerate(pbar):
    _ws = apply_augmentation(ws, sample_rate=SRATE)
    for i in range(_ws.shape[0]):
        _p = os.path.join(path, f"as{j}_{i}.wav")
        torchaudio.save(_p, _ws[i], SRATE)
        # print(f'saved in {_p}')
    pbar.set_postfix()

100%|██████████| 4519/4519 [1:25:13<00:00,  1.13s/batch]
